In [1]:
print("All Ok")

All Ok


# Docling PDF Processing Pipeline

```text
Input PDF
    ↓
Docling Configuration
    ↓
OCR + Layout Analysis + Table Detection
    ↓
DoclingDocument
    ├── Text
    ├── Headings
    ├── Tables
    ├── Pictures
    ├── Page Information
    └── Reading Order
    ↓
Output Formats
    ├── JSON
    ├── Markdown
    ├── Text
    ├── CSV
    └── Images

uv pip install docling langchain-docling tabulate

In [2]:
import json
from pathlib import Path

OUTOUT_DIR\ "file.json"

In [3]:
from docling.datamodel.base_models import InputFormat

In [4]:
InputFormat.PDF

<InputFormat.PDF: 'pdf'>

Complete PDF parsing pipeline configure for the Table extraction behaviour configure for Tesseract OCR(for fetcing the data from the images)

In [5]:
from docling.datamodel.pipeline_options import (
    PdfPipelineOptions,
    TableStructureOptions,
    TesseractCliOcrOptions,
)

Main Docling parser/converter

In [6]:
from docling.document_converter import (
    DocumentConverter,
    PdfFormatOption,
)

In [7]:
from docling_core.types.doc import (
    ImageRefMode,
    PictureItem,
    TableItem,
)

In [8]:
PDF_PATH = Path(
    r"data\complex_rag_parsing_sample_with_sunny_image.pdf"
)

OUTPUT_DIR = Path(
    r"data\parsed_complex_pdf_output"
)

In [9]:
PAGE_IMAGE_DIR = OUTPUT_DIR / "page_images"
PICTURE_DIR = OUTPUT_DIR / "extracted_pictures"
TABLE_DIR = OUTPUT_DIR / "extracted_tables"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PAGE_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
PICTURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

if not PDF_PATH.exists():
    raise FileNotFoundError(f"PDF not found: {PDF_PATH}")

In [10]:
pipeline_options = PdfPipelineOptions()

In [11]:
# Enable OCR
pipeline_options.do_ocr = True

In [12]:
pipeline_options.ocr_options = TesseractCliOcrOptions(
    tesseract_cmd=r"C:\Program Files\Tesseract-OCR\tesseract.exe",
    lang=["eng"],

    # False = selectable text + OCR where required
    # True = OCR complete pages
    force_full_page_ocr=False,
)

In [13]:
# Enable table structure extraction
pipeline_options.do_table_structure = True

In [14]:
pipeline_options.table_structure_options = TableStructureOptions(
    do_cell_matching=True,
)

Name Score Sunny 90 Anil 95

Row 1, Column 1 → Name Row 1, Column 2 → Score Row 2, Column 1 → Sunny Row 2, Column 2 → 90

In [15]:
# Preserve page and picture images
pipeline_options.images_scale = 2.0
pipeline_options.generate_page_images = True
pipeline_options.generate_picture_images = True

In [16]:
converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(
            pipeline_options=pipeline_options
        )
    }
)

In [17]:
pip install -U huggingface_hub

Note: you may need to restart the kernel to use updated packages.


c:\Users\Amanm\Desktop\Gen-AI Learning\env\Scripts\python.exe: No module named pip


In [18]:
pip install -U hf_xet

Note: you may need to restart the kernel to use updated packages.


c:\Users\Amanm\Desktop\Gen-AI Learning\env\Scripts\python.exe: No module named pip


In [19]:
print("Parsing PDF with Docling...")

conversion_result = converter.convert(PDF_PATH)

Parsing PDF with Docling...


The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
The plugin langchain_docling will not be loaded because Docling is being executed with allow_external_plugins=false.
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` 

2026-10-05 19:28:05,284 MatchingPostProcessor WARNING  2 of 35 pdf cells matched neither a row nor a column band of the 3x3 grid and were dropped from the table


OSD failed (doc data\complex_rag_parsing_sample_with_sunny_image.pdf, page: 0, OCR rectangle: 0, processed image file <tempfile._TemporaryFileWrapper object at 0x0000017CC850B7D0>):
 b'Estimating resolution as 220\r\nWarning. Invalid resolution 0 dpi. Using 70 instead.\r\nToo few characters. Skipping this page\r\nError during processing.\r\n'


In [20]:
docling_document = conversion_result.document

print("PDF parsed successfully.")
print("Total pages:", len(docling_document.pages))
print("Total tables:", len(docling_document.tables))
print("Total pictures:", len(docling_document.pictures))

PDF parsed successfully.
Total pages: 24
Total tables: 23
Total pictures: 10


In [21]:
json_output_path = OUTPUT_DIR / "docling_document.json"

In [22]:
json_output_path.write_text(
    json.dumps(
        docling_document.export_to_dict(),
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

8948094

In [23]:
print("JSON saved:", json_output_path)

JSON saved: data\parsed_complex_pdf_output\docling_document.json


In [24]:
markdown_output_path = OUTPUT_DIR / "rag_ready_document.md"

In [25]:
docling_document.save_as_markdown(
    markdown_output_path,
    image_mode=ImageRefMode.REFERENCED,
)

print("Markdown saved:", markdown_output_path)

Markdown saved: data\parsed_complex_pdf_output\rag_ready_document.md


In [26]:
text_output_path = OUTPUT_DIR / "extracted_text.txt"

text_output_path.write_text(
    docling_document.export_to_markdown(strict_text=True),
    encoding="utf-8",
)

Parameter `strict_text` has been deprecated and will be ignored.


33583

In [27]:
for page_number, page in docling_document.pages.items():

    if page.image is None:
        continue

    page_image_path = (
        PAGE_IMAGE_DIR
        / f"page_{page.page_no:03d}.png"
    )

    page.image.pil_image.save(
        page_image_path,
        format="PNG",
    )

In [28]:
print("Page images saved:", PAGE_IMAGE_DIR)

Page images saved: data\parsed_complex_pdf_output\page_images


In [29]:
for table_index, table in enumerate(
    docling_document.tables,
    start=1,
):
    table_df = table.export_to_dataframe(
        doc=docling_document
    )

    csv_path = TABLE_DIR / f"table_{table_index:03d}.csv"
    markdown_path = TABLE_DIR / f"table_{table_index:03d}.md"
    image_path = TABLE_DIR / f"table_{table_index:03d}.png"

    table_df.to_csv(
        csv_path,
        index=False,
        encoding="utf-8",
    )

    markdown_path.write_text(
        table_df.to_markdown(index=False),
        encoding="utf-8",
    )

    table_image = table.get_image(docling_document)

    if table_image is not None:
        table_image.save(image_path, format="PNG")

print("Tables saved:", TABLE_DIR)

Tables saved: data\parsed_complex_pdf_output\extracted_tables


In [30]:
picture_counter = 0

for element, _level in docling_document.iterate_items():

    if not isinstance(element, PictureItem):
        continue

    picture_counter += 1

    picture_path = (
        PICTURE_DIR
        / f"picture_{picture_counter:03d}.png"
    )

    picture_image = element.get_image(docling_document)

    if picture_image is not None:
        picture_image.save(
            picture_path,
            format="PNG",
        )

print("Pictures saved:", PICTURE_DIR)

Pictures saved: data\parsed_complex_pdf_output\extracted_pictures
